spark 에 hadoop map-reduce를 내포하고 있다. 빠르다.

In [1]:
import pyspark

import sys, os
os.environ["PYSPARK_PYTHON"] = r"D:\anaconda3\envs\analyzer\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"D:\anaconda3\envs\analyzer\python.exe"

In [2]:
pyspark.__version__

'3.5.1'

In [3]:
import subprocess
                                                   
# java -version 명령 실행
result = subprocess.run(["java", "-version"], capture_output=True, text=True, shell=True)
print(result.stderr)  # java -version은 stderr로 출력됨

openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment Temurin-17.0.18+8 (build 17.0.18+8)
OpenJDK 64-Bit Server VM Temurin-17.0.18+8 (build 17.0.18+8, mixed mode, sharing)



In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [5]:
spark = SparkSession.builder\
        .appName("PySpark Save Example")\
        .master("local[*]")\
        .getOrCreate()

spark_df = spark.read.csv("data/Subjects.csv", header = True, inferSchema = True)
spark_df = spark_df.withColumn("total", col("kor") + col("eng") + col("math") + col("science"))
spark_df.show()


+-----+----+---+---+----+-------+-----+
|class|name|kor|eng|math|science|total|
+-----+----+---+---+----+-------+-----+
|    1| aaa| 67| 87|  90|     98|  342|
|    1| bbb| 45| 45|  56|     98|  244|
|    1| ccc| 95| 59|  96|     88|  338|
|    1| ddd| 65| 94|  89|     98|  346|
|    1| eee| 45| 65|  78|     98|  286|
|    1| fff| 78| 76|  98|     89|  341|
|    2| ggg| 87| 67|  65|     56|  275|
|    2| hhh| 89| 98|  78|     78|  343|
|    2| iii|100| 78|  56|     65|  299|
|    2| jjj| 99| 89|  87|     87|  362|
|    2| kkk| 98| 45|  56|     54|  253|
|    2| lll| 65| 89|  87|     78|  319|
+-----+----+---+---+----+-------+-----+



In [6]:
from pyspark.sql.functions import col

# 컬럼 추가 : 총점
spark_df = spark_df.withColumn("total", col("kor") + col("eng") + col("math") + col("science"))
spark_df.show()

# 평균점수 계산
spark_df.groupby().avg("total").show()

# 조건 필터링 : 총점 300점 이상
spark_df.filter(col("total") >= 300).show()


# Action

+-----+----+---+---+----+-------+-----+
|class|name|kor|eng|math|science|total|
+-----+----+---+---+----+-------+-----+
|    1| aaa| 67| 87|  90|     98|  342|
|    1| bbb| 45| 45|  56|     98|  244|
|    1| ccc| 95| 59|  96|     88|  338|
|    1| ddd| 65| 94|  89|     98|  346|
|    1| eee| 45| 65|  78|     98|  286|
|    1| fff| 78| 76|  98|     89|  341|
|    2| ggg| 87| 67|  65|     56|  275|
|    2| hhh| 89| 98|  78|     78|  343|
|    2| iii|100| 78|  56|     65|  299|
|    2| jjj| 99| 89|  87|     87|  362|
|    2| kkk| 98| 45|  56|     54|  253|
|    2| lll| 65| 89|  87|     78|  319|
+-----+----+---+---+----+-------+-----+

+-----------------+
|       avg(total)|
+-----------------+
|312.3333333333333|
+-----------------+

+-----+----+---+---+----+-------+-----+
|class|name|kor|eng|math|science|total|
+-----+----+---+---+----+-------+-----+
|    1| aaa| 67| 87|  90|     98|  342|
|    1| ccc| 95| 59|  96|     88|  338|
|    1| ddd| 65| 94|  89|     98|  346|
|    1| fff| 78| 7

In [7]:
df.groupby().avg("total")

DataFrame[avg(total): double]

In [8]:
# spark SQL 사용
# Temp view 생성 (쿼리 사용하려면 템프 뷰 만들어서 쿼리 사용해야함.) 변환 안 하면 실행 안될 듯?
df.createOrReplaceTempView('students')


# spark.sql(...):
# - SparkSession 객체를 통해 SQL 쿼리 실행
# - 일반적인 RDBMS의 SQL과 거의 동일하게 사용 가능
# - 실행 결과는 다시 DataFrame형태로 반환됨

# SQL 쿼리
high_score = spark.sql("SELECT name, total FROM students WHERE total >= 300")
high_score.show()

+----+-----+
|name|total|
+----+-----+
| aaa|  342|
| ccc|  338|
| ddd|  346|
| fff|  341|
| hhh|  343|
| jjj|  362|
| lll|  319|
+----+-----+



In [9]:
spark.stop()
print("SparkSession 종료")

SparkSession 종료


In [10]:
# PySpark 데이터 분석 + 머신러닝 (ML lib) 실습 예제

In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.classification import DecisionTreeClassifier

In [12]:
# SparkSession 생성

spark = SparkSession.builder\
    .appName("PySpark_ML_Example")\
    .master("local[*]")\
    .getOrCreate()

print("SparkSession 생성 완료!")

SparkSession 생성 완료!


In [15]:
# csv파일 로드

df = spark.read.csv("./data/Subjects.csv",
                    header = True, # header= True --> 첫 번째 행을 컬럼명으로 사용
                    inferSchema = True # 데이터 타입 자동 추론
)

print("csv 파일 로드 완료!")
df.show()

csv 파일 로드 완료!
+-----+----+---+---+----+-------+
|class|name|kor|eng|math|science|
+-----+----+---+---+----+-------+
|    1| aaa| 67| 87|  90|     98|
|    1| bbb| 45| 45|  56|     98|
|    1| ccc| 95| 59|  96|     88|
|    1| ddd| 65| 94|  89|     98|
|    1| eee| 45| 65|  78|     98|
|    1| fff| 78| 76|  98|     89|
|    2| ggg| 87| 67|  65|     56|
|    2| hhh| 89| 98|  78|     78|
|    2| iii|100| 78|  56|     65|
|    2| jjj| 99| 89|  87|     87|
|    2| kkk| 98| 45|  56|     54|
|    2| lll| 65| 89|  87|     78|
+-----+----+---+---+----+-------+



In [16]:
# 데이터 전처리 및 파생변수 생성
# - 결측값은 0으로 대체
# - 총점 ( total), 합격여부 (pass) 파생 변수 생성

df = df.fillna({"kor": 0, "eng": 0, "math": 0, "science": 0})
df = df.withColumn("total" , col("kor") + col("eng") + col("math") + col("science"))
df = df.withColumn("pass", when(col("total") >= 300, 1).otherwise(0))

print("데이터 전처리 완료")
df.show()

데이터 전처리 완료
+-----+----+---+---+----+-------+-----+----+
|class|name|kor|eng|math|science|total|pass|
+-----+----+---+---+----+-------+-----+----+
|    1| aaa| 67| 87|  90|     98|  342|   1|
|    1| bbb| 45| 45|  56|     98|  244|   0|
|    1| ccc| 95| 59|  96|     88|  338|   1|
|    1| ddd| 65| 94|  89|     98|  346|   1|
|    1| eee| 45| 65|  78|     98|  286|   0|
|    1| fff| 78| 76|  98|     89|  341|   1|
|    2| ggg| 87| 67|  65|     56|  275|   0|
|    2| hhh| 89| 98|  78|     78|  343|   1|
|    2| iii|100| 78|  56|     65|  299|   0|
|    2| jjj| 99| 89|  87|     87|  362|   1|
|    2| kkk| 98| 45|  56|     54|  253|   0|
|    2| lll| 65| 89|  87|     78|  319|   1|
+-----+----+---+---+----+-------+-----+----+



In [17]:
# 내용적으로는 좋지 않지만 아래 구조를 중점적으로 보자.
# 회귀 모델 LinearRegression
# - 입력 피쳐 : kor, eng, math, science <-- x 인자
# - 타겟 : total <-- y 인자
# - 분석 : 새로운 x 인자를 받아서 총점 (total)을 예측하는 회귀모델

assembler = VectorAssembler( # 스파크에서는 벡터형태로 데이터를 맞춰줘야 함.
    inputCols=["kor", "eng", "math", "science"],
    outputCol="features"
)
# 스파크는 하나의 벡터로 연결

train_df = assembler.transform(df).select("features", "total") # x, y set vectorization.0

# 회귀모델 생성 및 학습
lr = LinearRegression(featuresCol="features", labelCol="total")
lr_model = lr.fit(train_df)

# 예측 수행
'''
transform(df) --> 학습한 모델로 df의 y(label) 예측
'''
lr_predictions = lr_model.transform(train_df)
print("Linear Regression 모델 학습 완료")

lr_predictions.select( "features","total","prediction").show(5)


Linear Regression 모델 학습 완료
+--------------------+-----+------------------+
|            features|total|        prediction|
+--------------------+-----+------------------+
|[67.0,87.0,90.0,9...|  342| 342.0000000000004|
|[45.0,45.0,56.0,9...|  244|243.99999999999918|
|[95.0,59.0,96.0,8...|  338|338.00000000000153|
|[65.0,94.0,89.0,9...|  346| 346.0000000000002|
|[45.0,65.0,78.0,9...|  286|285.99999999999903|
+--------------------+-----+------------------+
only showing top 5 rows



In [ ]:
# 학습 결과, 성능 확인, 튜닝 등의 작업은 스킵. 그러니까 알아서 공부 해야함.

In [18]:
# 분류 모델 (Logistic Regression)

# 입력 피쳐 : kor, eng, math, science <-- x인자
# - 타겟 : pass (합격 여부 ) <-- y 인자
# - 분석 : x 인자를 입력으로 모델이 합격 (1) / 불합격 (1) 을 분류

In [22]:
assembler2 = VectorAssembler(
    inputCols=["kor", "eng", "math", "science"],
    outputCol="features2"
)

train_df2 = assembler2.transform(df).select("features2", "pass") # x, y set

# 분류 모델 생성 및 학습
logr = LogisticRegression(featuresCol="features2", labelCol="pass")
logr_model = logr.fit(train_df2)

# 예측 수행
logr_predictions = logr_model.transform(train_df2)

print("Logistic Regression 모델 학습 완료")
logr_predictions.select("features2","pass","prediction","probability").show(5, truncate=False)

spark.stop()

Logistic Regression 모델 학습 완료
+---------------------+----+----------+------------------------------------------+
|features2            |pass|prediction|probability                               |
+---------------------+----+----------+------------------------------------------+
|[67.0,87.0,90.0,98.0]|1   |1.0       |[2.81944823643108E-9,0.9999999971805518]  |
|[45.0,45.0,56.0,98.0]|0   |0.0       |[1.0,0.0]                                 |
|[95.0,59.0,96.0,88.0]|1   |1.0       |[1.245296963688586E-8,0.9999999875470303] |
|[65.0,94.0,89.0,98.0]|1   |1.0       |[1.852718822498173E-10,0.9999999998147281]|
|[45.0,65.0,78.0,98.0]|0   |0.0       |[0.9999999769882363,2.3011763716773714E-8]|
+---------------------+----+----------+------------------------------------------+
only showing top 5 rows



# transformer 모델은 블랙박스이기 때문에 설명력이 많이 떨어짐.
# --> 실제 분석 시 transformer 모델 사용은 지양할 예정

설명 가능한 컬럼 하나 추가해서 shap value 를 추가

품질 불량 분류 할 때, 이미지 분석을 통해서, 불량 판단할 때 분류 사용. // 실제 이것까지만 가지고는 문제를 찾아서 조치하기 애매하고.
불량의 클래스를 더 딥하게 정의 하고, 어떤 종류의 불량인지를 더 상세하게 정의 해야함.

이 제품의 특징이 불량 / 양폼 구분하고
불량이라면 어떤 내용의 불량 원인인다.
도메인 담당자와 해당 클래스가 정확히 정의되어있는지 이야기하고 하나하나 정의하기.

In [24]:
#머신러닝을 위한 건설근로자 중대사고 위험성 예측 모델
#https://chatgpt.com/s/t_69c63dd44d7c8191a7dce5a78efc22eb

SyntaxError: unterminated string literal (detected at line 1) (966828687.py, line 1)

In [26]:
# gpt 대화
# https://chatgpt.com/s/t_69c63dd44d7c8191a7dce5a78efc22eb
# https://chatgpt.com/s/t_69c63e6e1b988191aef50d48659ddbcb

#OpenFoaM
# https://www.openfoam.com/download/openfoam-installation-on-windows-10